# Module 03: EDA

In [ ]:
# packages
import numpy as np 
import matplotlib.pyplot as plt
from matplotlib.pyplot import subplots
from sklearn.model_selection import train_test_split 
from ISLP import load_data

# set seed
seed = 2323

### We'll use the _Hitters_ data from ISLP for this activity. The metadata for _Hitters_ can be found [here](https://intro-stat-learning.github.io/ISLP/datasets/Hitters.html).

In [ ]:
# Load the data
Hitters = load_data('Hitters')

### Determine the number of rows and columns in the dataset by returning its "shape" attribute

In [ ]:
Hitters.shape

### Determine whether each feature is numeric or categorical by returning the "dtype" attribute for each column

In [ ]:
for col in Hitters.columns:
    if Hitters[col].dtype == int:
        print(f"{col} is Numeric")
    elif Hitters[col].dtype == float:
        print(f"{col} is Numeric")
    else:
        print(f"{col} is Categorical")



### Before doing any other analyses, let's create training and test sets.

In [ ]:
Train, Test = train_test_split(Hitters, 
                               random_state=seed, 
                               test_size=0.40, 
                               shuffle=True) 

### Based on the metadata, what is the difference between the 6 columns starting with 'C' and the 6 related columns that don't?

The C columns are career totals, and are therefore much higher than their related columns which are just for the given year unless it is the players first year in the league.

### On the training set, create pairwise scatterplots for each of these 6 columns with the 'Salary' variable.

In [ ]:
# First create a subset of the columns that we want to plot

subset = Train[['CAtBat', 'CHits', 'CHmRun', 'CRuns', 'CRBI', 'CWalks']]

# Initialize the plots before drawing them

fig, axes = subplots(nrows=2,
                     ncols=3,
                     figsize=(15, 10))
# Copy the helper function

def range_to_grid(i,Ncol):
    x=[]
    y=[]
    for n in range(Ncol**2):
        x.append(int(np.floor(n/Ncol)))
        y.append(n % Ncol)
        #print(n,x[n],y[n])
    return x[i],y[i]

# Plot the variables

for j in range(len(subset.columns)):
    axes[range_to_grid(j,3)[0],range_to_grid(j,3)[1]].plot(subset.iloc[:,j], Train['Salary'], 'o')
    axes[range_to_grid(j,3)[0],range_to_grid(j,3)[1]].set_xlabel(subset.columns[j])


### Use the "describe" method to determine the mean, standard deviation, and 5 number summary of all numeric variables in the training subset of _Hitters_.

In [ ]:
Train.describe()

### It looks like the mean and median of 'AtBat' are nearly equal. This _might_ suggest that this variable is normally distributed. Create a histogram of 'AtBat' to check this hypothesis.

In [ ]:
at_bat_data = Train["AtBat"].dropna()

fig, ax = subplots(figsize=(8, 5))

ax.hist(at_bat_data, bins=20, color="blue", edgecolor="black")

ax.set_title("Distribution of 'AtBat'")
ax.set_xlabel("Number of At Bats")
ax.set_ylabel("Frequency")
ax.grid(axis="y", linestyle="--", alpha=0.7)

### Let's standardize the AtBat feature (i.e., normalize by z-scores). We'll create a new column in the training data called 'AtBat_st' to represent this.

In [ ]:
Train['AtBat_st'] = (Train['AtBat'] - Train["AtBat"].mean()) / Train["AtBat"].std()

### How many rows have an 'AtBat' value within the first standard deviation?

Hint: the 'len' magic method returns the number of rows of a dataFrame.

In [ ]:
len(Train[(Train['AtBat_st'] < 1) & (Train['AtBat_st'] > -1)])

### Going back to the results of the 'describe' method, how can you tell that the 'Salary' variable has missing values?

All of the other variables have a count of 193, while the salary variable has a count of 164, which shows that 29 observations in the training data are missing salary values.

### Describe a situation where a variable could have missing values but this would not be reflected in the results of the 'describe' method.

The missing value may have a placeholder, like 0 or -999, that allows the describe function to include it without a meaningful value being present. This could also skew other parts of the descirbe function, like the mean, making the function unreliable overall.

### On the training data, create separate boxplots of the 'AtBat' variable for when 'Salary' is populated or missing.

In [ ]:
#fill in
salary = Train[Train["Salary"].notnull()]
nonsalary = Train[Train["Salary"].isnull()]

fig, ax = plt.subplots()
ax.boxplot([salary["AtBat"], 
            nonsalary["AtBat"]], 
           positions = [1, 2])
ax.set_title("Boxplot of AtBat")
ax.set_ylabel("AtBat")
ax.set_xticklabels(["Salary Populated", "Salary Missing"])
ax.set(xlim=(0.5, 2.5))




### Create a correlation matrix for all numeric features in the training set

In [ ]:
numeric_train = Train.select_dtypes(include=[np.number])
corr_matrix = numeric_train.corr()
corr_matrix


### Propose two different ways of imputing the missing values of Salary while taking advantage of the information given in the boxplots or the correlation matrix.

One idea would be to fill in the salary value in missing places with the average of salaries in a given bucket. I would recommend using one the C stats to make the buckets because they have the highest correlation to salary in the correlation matrix. 

Another idea would be to create a multiple linear regression model to try and predict where thes observations with missing salaries would fit into the data using the variables present for every observation.

### For our last exercise, we'll explore Hits and Walks relative to AtBat totals. 
- Use the sum function to calculuate the totals of each of these three variables for the 1986 season (on the training set). 
- Create a pie chart which shows total hits, total walks, and remaining total (neither) as percents of the At Bats total (on the training set). 

In [ ]:
TotHits = sum(Train['CHits'])
TotWalks = sum(Train['CWalks'])
TotAtBat = sum(Train['CAtBat'])

Labels = ['Hits', 'Walks', 'Neither']
Totals = [TotHits, TotWalks, TotAtBat-TotHits-TotWalks]

In [ ]:
# pie chart
plt.pie(Totals, labels=Labels, autopct='%1.1f%%', startangle=90)
plt.show()

### The previous two cells gave us totals across all players. For each player in the training set, calculate the Hits as a percent of AtBat and store it in a new variable called 'AVG'

In [ ]:
#fillin
Train['AVG'] = Train['CHits'] / Train['CAtBat']

### Using 0.25 and 0.31 as the split points, create a new variable with three bins: high, medium, and low. 

In [ ]:
Train['AVG_bin'] = 'medium'
Train['AVG_bin'] = np.where(Train['AVG'] < 0.25, 'low', Train['AVG_bin'])
Train['AVG_bin'] = np.where(Train['AVG'] > 0.31, 'high', Train['AVG_bin'])


### Create a bar chart that displays the number of players in each of the low, medium, and high categories (for the training data).

In [ ]:
Train['AVG_bin'].value_counts()

plt.bar(Train['AVG_bin'].value_counts().index, Train['AVG_bin'].value_counts().values)
plt.ylabel('Players in Each Batting Average Category')
plt.xlabel('Batting Average Category')
plt.title('Distribution of Players by Batting Average')
plt.show()

Notice that the order of the bars will be medium, low, high. That's counterintuitive. We can reorder these quickly. 

In [ ]:
indexMap = ['low', 'medium', 'high']
reordered_list = [Train['AVG_bin'].value_counts()[i] for i in indexMap]

In [ ]:
plt.bar(indexMap, reordered_list)

plt.title("1986 AVG (Training Set)")
plt.ylabel("Number of Players")

plt.xticks(indexMap)

plt.show()

### Did we use the depth method or width method for creating these bins? Explain.

This is a width model. The main characteristic of a depth model is that there is the same or very similar amounts of observations in each bin. This model has over 120 observations in the 'medium' bin, around 55 in the 'low' bin and about 5 in the 'high' bin. These are obviously very different amount, therefore this is a width model. Also, if we look at the describe function after adding the batting average variable, the minimum is .19 and the max is .35. Therefore, by putting the bin sizes at .25 and .31, we have covered roughly .05 with each bin.